In [28]:
import os
if os.path.basename(os.getcwd()) == "limpieza":
    os.chdir("..")
    
import pandas as pd
import numpy as np
from utils.funciones_filtrado import tipo_nulo_unicos_x_columna

In [29]:
base_inventario = pd.read_excel('inventario.xlsx')
base_inventario.head()

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,P001,Americano,Café Veracruz SA,120.0,30,8.5,2024-11-01
1,P002,Latte,Café Veracruz SA,95.0,25,9,2024-10-15
2,P003,Cappuccino,Café Veracruz SA,NaN,20,8.75,2024-09-20
3,P004,Espresso,Café Veracruz SA,200.0,50,6,2024-11-10
4,P005,frappe chocolate,Bebidas Frías MX,60.0,15,12,2024-10-01


Lo primero que vamos a realizar es quitar esa fila, donde habiamos notado que un porducto se repetia, que es el 'Americano' del proveedor 'Café Veracruz SA', pero quitaremos el registro con el id 'P016' ya que notamos que en su costo es el que está escrito con letra, además que su última compra ya tiene tiempo, porque entre la más reciente y este registro tienen una diferencia de 4 meses. Además vamos a asginar un id al producto que no tiene.
Y por último el valor faltante de la columna 'stock_atual' se llenará con 0, debido a que como no tenemos información, es mejor que se ponga que no hay y hacer el pedido a que se queden sin el producto.

In [30]:
base_inventario.drop(base_inventario[base_inventario['producto_id']=='P016'].index, inplace=True)
# de esta forma se está llenando ya que aprovechamos que sabemos que sólo hay un valor nulo, ya que para 
# otras situaciones hay que adoptar métodos más precisos para saber que se está llenando
base_inventario['producto_id'] = base_inventario['producto_id'].fillna('P017')
base_inventario['stock_actual'] = base_inventario['stock_actual'].fillna(0)
base_inventario['costo'] = base_inventario['costo'].astype('float64')
base_inventario['ultima_compra'] = pd.to_datetime(base_inventario['ultima_compra'])
base_inventario 

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,P001,Americano,Café Veracruz SA,120.0,30,8.50,2024-11-01
1,P002,Latte,Café Veracruz SA,95.0,25,9.00,2024-10-15
2,P003,Cappuccino,Café Veracruz SA,0.0,20,8.75,2024-09-20
3,P004,Espresso,Café Veracruz SA,200.0,50,6.00,2024-11-10
4,P005,frappe chocolate,Bebidas Frías MX,60.0,15,12.00,2024-10-01
5,P006,Frappe Caramelo,Bebidas Frías MX,55.0,15,12.50,2024-10-01
6,P007,te verde,Infusiones del Sur,80.0,20,4.50,2024-08-15
7,P008,Muffin Arandano,Panadería Local,40.0,10,11.00,2024-11-05
8,P009,Muffin Chocolate,Panadería Local,35.0,10,11.00,2024-11-05
9,P010,Croissant,Panadería Local,25.0,8,-5.50,2024-10-20


Gracias a que la lista del inventario es corta se pudo notar que la columna de costo hay un valor negativo, lo cual no tendría sentido que un producto tendría un costo negativo, ya que en ese caso en lugar de comprarlo es como que aparte de que nos dan el producto nos dan dinero. Por lo que se hará el cambio correspondiente a un positivo.
De esta forma también habría que verficiar que ciertas columnas tuvieran valores positivos.

In [31]:
base_inventario['costo'] = np.where(base_inventario['costo']<0, base_inventario['costo']*-1, base_inventario['costo'])
base_inventario

,producto_id,producto,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,P001,Americano,Café Veracruz SA,120.0,30,8.50,2024-11-01
1,P002,Latte,Café Veracruz SA,95.0,25,9.00,2024-10-15
2,P003,Cappuccino,Café Veracruz SA,0.0,20,8.75,2024-09-20
3,P004,Espresso,Café Veracruz SA,200.0,50,6.00,2024-11-10
4,P005,frappe chocolate,Bebidas Frías MX,60.0,15,12.00,2024-10-01
5,P006,Frappe Caramelo,Bebidas Frías MX,55.0,15,12.50,2024-10-01
6,P007,te verde,Infusiones del Sur,80.0,20,4.50,2024-08-15
7,P008,Muffin Arandano,Panadería Local,40.0,10,11.00,2024-11-05
8,P009,Muffin Chocolate,Panadería Local,35.0,10,11.00,2024-11-05
9,P010,Croissant,Panadería Local,25.0,8,5.50,2024-10-20


In [32]:
inspeccion = tipo_nulo_unicos_x_columna(base_inventario)
inspeccion

,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,producto_id,object,0,16
1,producto,object,0,16
2,proveedor,object,0,7
3,stock_actual,float64,0,15
4,stock_minimo,int64,0,10
5,costo,float64,0,13
6,ultima_compra,datetime64[ns],0,11


Pero recordemos que la tabla productos también tiene 2 columnas similares, por lo que vamos a revisar si ambas columnas tienen cierta relación o no. Para ello es necesario traer a tabla de productos limpia.

In [33]:
base_productos = pd.read_excel('productos_limpio.xlsx')
#Hacemos momentaneamente un cambio de nombre de columnas para no confundir la tabla de procedencia
base_inventario = base_inventario.rename(columns={'producto_id':'producto_id_i', 'producto':'producto_i'}) 
revision = pd.concat([base_inventario, base_productos], axis=1)
revision[['producto_id_i', 'producto_i','producto_id', 'producto']]

,producto_id_i,producto_i,producto_id,producto
0,P001,Americano,P001,Americano
1,P002,Latte,P002,Latte
2,P003,Cappuccino,P003,Latte Vainilla
3,P004,Espresso,P004,Cappuccino
4,P005,frappe chocolate,P005,Espresso
5,P006,Frappe Caramelo,P006,Frappé Chocolate
6,P007,te verde,P007,Frappe Caramelo
7,P008,Muffin Arandano,P008,Té Verde
8,P009,Muffin Chocolate,P009,Té Negro
9,P010,Croissant,P010,Muffin Arándano


Notamos que los productos del inventario y los que se venden, no tienen mucha relación debido a que no comparten el mismo id, aunque podría ser igual en algunos casos pero no en todos, además que en ambas tablas no están los mismos productos, y esto cobra sentido, debido a que los productos que se venden necesitan ciertos insumos para ser preparados, por el ejemplo el frappe lleva leche, por lo que de insumos necesitaría leche y un tipo de cafe, por lo que haría falta un cambio de nombres a las columnas y un cambio de id.

In [34]:
base_inventario = base_inventario.rename(columns={'producto_id_i':'insumo_id', 'producto_i':'insumo'}) 
base_inventario['insumo_id'] = base_inventario['insumo_id'].str.replace('P', 'I')
base_inventario

,insumo_id,insumo,proveedor,stock_actual,stock_minimo,costo,ultima_compra
0,I001,Americano,Café Veracruz SA,120.0,30,8.50,2024-11-01
1,I002,Latte,Café Veracruz SA,95.0,25,9.00,2024-10-15
2,I003,Cappuccino,Café Veracruz SA,0.0,20,8.75,2024-09-20
3,I004,Espresso,Café Veracruz SA,200.0,50,6.00,2024-11-10
4,I005,frappe chocolate,Bebidas Frías MX,60.0,15,12.00,2024-10-01
5,I006,Frappe Caramelo,Bebidas Frías MX,55.0,15,12.50,2024-10-01
6,I007,te verde,Infusiones del Sur,80.0,20,4.50,2024-08-15
7,I008,Muffin Arandano,Panadería Local,40.0,10,11.00,2024-11-05
8,I009,Muffin Chocolate,Panadería Local,35.0,10,11.00,2024-11-05
9,I010,Croissant,Panadería Local,25.0,8,5.50,2024-10-20


In [35]:
base_inventario.to_excel('inventario_limpio.xlsx', index=False)